In [ ]:
# ================================================
# Decision Tree Induction (Han, Pei & Tong, 2022)
# Works with ANY generalized dataset uploaded as CSV
# Just upload to Colab and set `file_path`
# ================================================

import pandas as pd
import numpy as np

# ----------------------------------------
# 1. Load your generalized dataset from CSV
# ----------------------------------------
file_path = "/content/generalized_data.csv"   # <-- update this to your file name
df = pd.read_csv(file_path)

print(f"Dataset loaded: {df.shape}")
display(df.head())

# --------------------------------------------------------
# 2. Compute weighted class totals at the root of the tree
# --------------------------------------------------------
root_counts = df.groupby("status")["count"].sum()
print("\nRoot Node Weighted Counts:")
print(root_counts)

# -----------------------------------------
# 3. Define Gini impurity and Info Gain
# -----------------------------------------
def gini(counts):
    total = sum(counts)
    return 1 - sum((c/total)**2 for c in counts)

def info_gain(df, attribute):
    root_impurity = gini(df.groupby("status")["count"].sum())
    total = df["count"].sum()

    weighted_gini = 0
    for val, subset in df.groupby(attribute):
        subset_counts = subset.groupby("status")["count"].sum()
        weighted_gini += (subset["count"].sum() / total) * gini(subset_counts)

    return root_impurity - weighted_gini

# ---------------------------------------------------
# 4. Compute information gain for each candidate attribute
# ---------------------------------------------------
attributes = ["department", "age", "salary"]

gains = {attr: info_gain(df, attr) for attr in attributes}
print("\nInformation Gain for Each Attribute:")
for k, v in gains.items():
    print(f"{k}: {v:.6f}")

# ---------------------------------------------------
# 5. Select the root attribute (max information gain)
# ---------------------------------------------------
root_split = max(gains, key=gains.get)
print(f"\nRoot splitting attribute: {root_split}")

# ---------------------------------------------------
# 6. Partition by the root attribute (salary in your case)
# ---------------------------------------------------
salary_groups = {}

for sal, subset in df.groupby(root_split):
    salary_groups[sal] = subset.groupby("status")["count"].sum()

print("\nSalary Groups (Class Distributions):")
salary_groups

# ---------------------------------------------------
# 7. Identify pure and mixed salary partitions
# ---------------------------------------------------
pure_nodes = {}
mixed_nodes = {}

for sal, counts in salary_groups.items():
    if len(counts[counts > 0]) == 1:
        pure_nodes[sal] = counts
    else:
        mixed_nodes[sal] = counts

print("\nPure Salary Nodes:")
print(pure_nodes)

print("\nMixed Salary Nodes:")
print(mixed_nodes)

# ---------------------------------------------------
# 8. Process mixed node(s)
# ---------------------------------------------------
if mixed_nodes:
    for mixed_salary in mixed_nodes:
        print(f"\nProcessing mixed partition: {mixed_salary}")

        mixed_df = df[df[root_split] == mixed_salary]
        dept_split = mixed_df.groupby("department")["count"].sum().groupby(mixed_df["department"]).sum()

        print("\nDepartment Split for Mixed Node:")
        print(dept_split)

# ---------------------------------------------------
# 9. Build Table 1 automatically (decision tree summary)
# ---------------------------------------------------
# NOTE — Because this depends on your dataset structure,
# we dynamically generate the tree summary.

rows = []
node_id = 1

# ROOT NODE
rows.append([
    node_id, "Root", root_split, "—",
    f"({root_counts.get('senior',0)}, {root_counts.get('junior',0)})",
    f"Split on {root_split}"
])

# LEAVES + MIXED NODE HANDLING
leaf_counter = 1.1

for sal, counts in salary_groups.items():
    senior = counts.get("senior", 0)
    junior = counts.get("junior", 0)

    if sal in pure_nodes:
        decision = "senior" if senior > 0 else "junior"
        rows.append([leaf_counter, "Leaf", "—", f"{root_split} = {sal}", f"({senior}, {junior})", decision])
        leaf_counter = round(leaf_counter + 0.1, 1)

    else:
        # Mixed node → internal node
        rows.append([leaf_counter, "Internal Node", "department",
                     f"{root_split} = {sal}", f"({senior}, {junior})",
                     "Split on department"])
        parent_id = leaf_counter
        leaf_counter = round(leaf_counter + 0.1, 1)

        # Now add child leaves
        mixed_df = df[df[root_split] == sal]
        for dept, subset in mixed_df.groupby("department"):
            dept_counts = subset.groupby("status")["count"].sum()
            s = dept_counts.get("senior", 0)
            j = dept_counts.get("junior", 0)
            decision = "senior" if s > 0 else "junior"

            rows.append([
                float(f"{parent_id}{round(leaf_counter*10)%10}"),
                "Leaf",
                "—",
                f"department = {dept}",
                f"({s}, {j})",
                decision
            ])
            leaf_counter = round(leaf_counter + 0.1, 1)

# Build table
table = pd.DataFrame(rows, columns=[
    "Node ID","Node Type","Split Attribute","Branch / Condition",
    "Class Distribution (Senior, Junior)","Decision / Outcome"
])

print("\n==============================")
print("FINAL DECISION TREE TABLE")
print("==============================")
display(table)

# ---------------------------------------------------
# 10. Narrative summary
# ---------------------------------------------------
print("""
Decision Tree Induction Summary (Han, Pei, & Tong, 2022)

1. Root Node:
   - Weighted class counts computed from input dataset.
   - Node tested for purity — not pure → continue splitting.

2. Attribute Selection:
   - Information gain computed for department, age, and salary.
   - Attribute with highest information gain selected for root split.

3. Partitioning:
   - Salary partitions evaluated for class purity.
   - Pure groups become leaf nodes.
   - Mixed groups require further splitting.

4. Mixed Node:
   - Mixed salary groups evaluated by department.
   - Pure class distributions result in leaf nodes.

The final tree uses salary at the root, with department only inside mixed ranges.
""")